# 🥉 Ingestão Bronze Delta - Ecommerce Pedidos

## Squad 2 — Ecommerce

### Tabela: ecommerce_pedidos


## Objetivo

Realizar a ingestão incremental dos arquivos
`ecommerce_pedidos.parquet`
da camada Raw para Bronze utilizando Delta Lake.


## Origem

Azure Data Lake Gen2:

raw/real-time-data


## Destino

Azure Data Lake Gen2:

grupo3/bronze/ecommerce_pedidos


## Processamento realizado

- Leitura dos micro-lotes disponíveis;
- Controle de arquivos processados;
- Inclusão de metadados de auditoria;
- Particionamento temporal;
- Gravação Delta.



In [0]:
# ==================================================
# IMPORTAÇÃO DAS BIBLIOTECAS
# ==================================================

# Biblioteca principal utilizada para manipulação de DataFrames Spark,
# criação de colunas, funções de data e transformações necessárias
# durante o processo de ingestão Bronze.
from pyspark.sql import functions as F


# Biblioteca responsável pela autenticação no Azure utilizando
# Service Principal (Client ID, Tenant ID e Client Secret).
# Será utilizada para criar uma conexão segura com o ADLS Gen2.
from azure.identity import ClientSecretCredential


# Cliente oficial do Azure Data Lake Storage Gen2.
# Permite listar arquivos, acessar diretórios e validar
# estruturas existentes dentro dos containers.
from azure.storage.filedatalake import DataLakeServiceClient


# Biblioteca utilizada para leitura temporária dos arquivos Parquet.
# Necessária porque alguns arquivos possuem timestamps no formato
# TIMESTAMP(NANOS), que apresentam incompatibilidade com leitura
# direta pelo Spark em determinados ambientes.
import pandas as pd


# Permite transformar o conteúdo binário baixado do Azure
# em uma estrutura que o Pandas consegue interpretar como Parquet.
from io import BytesIO



# Validação simples para confirmar que todas as bibliotecas
# necessárias foram carregadas corretamente.
print(
    "Bibliotecas carregadas com sucesso!"
)

Bibliotecas carregadas com sucesso!


In [0]:
# ==================================================
# CONFIGURAÇÃO DE ACESSO AO AZURE DATA LAKE GEN2
# ==================================================


# Nome do Secret Scope utilizado para armazenar
# as credenciais de acesso ao Azure.
# As informações sensíveis não ficam expostas no notebook.
SECRET_SCOPE = "mercadata-ecommerce-pedidos"



# Nome do Storage Account onde estão os dados
# do projeto de Engenharia de Dados.
storage_account = "internshipdatalake"



# Recupera o Client ID da aplicação Azure
# armazenado de forma segura no Databricks Secret Scope.
client_id = dbutils.secrets.get(
    scope=SECRET_SCOPE,
    key="ADLS_CLIENT_ID"
)



# Recupera o Tenant ID do Azure AD.
# Identifica o ambiente Microsoft Azure
# onde a aplicação está registrada.
tenant_id = dbutils.secrets.get(
    scope=SECRET_SCOPE,
    key="ADLS_TENANT_ID"
)



# Recupera o Client Secret.
# Essa credencial permite autenticação da aplicação
# sem expor senha ou informações sensíveis.
client_secret = dbutils.secrets.get(
    scope=SECRET_SCOPE,
    key="ADLS_CLIENT_SECRET"
)



# ==================================================
# CONFIGURAÇÃO OAUTH PARA OPERAÇÕES SPARK
# ==================================================

# O Spark utiliza essas configurações para autenticar
# diretamente no Azure Data Lake Gen2 utilizando
# OAuth com Service Principal.
#
# Essa configuração será passada nas operações:
#
# spark.read.options(**adls_options)
#
# e
#
# df.write.options(**adls_options)

adls_options = {

    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net":
        "OAuth",


    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",


    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net":
        client_id,


    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net":
        client_secret,


    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}



# ==================================================
# CRIAÇÃO DO CLIENTE AZURE SDK
# ==================================================

# Cria uma credencial utilizando o Service Principal.
# Essa autenticação será utilizada pelo SDK do Azure
# para acessar os containers.
credential = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret
)



# Cria o cliente principal do Data Lake.
# A partir dele conseguimos acessar diferentes containers.
service_client = DataLakeServiceClient(
    account_url=(
        f"https://{storage_account}.dfs.core.windows.net"
    ),
    credential=credential
)



# Cliente para acessar o container RAW.
# Local onde chegam os arquivos Parquet dos micro-lotes.
raw_fs_client = (
    service_client
    .get_file_system_client(
        "raw"
    )
)



# Cliente para acessar o container SQUAD2.
# Local onde será gravada a camada Bronze.
squad2_client = (
    service_client
    .get_file_system_client(
        "squad2"
    )
)



print(
    "Conexão Azure configurada com sucesso!"
)

Conexão Azure configurada com sucesso!


In [0]:
# ==================================================
# DEFINIÇÃO DOS CAMINHOS DO PIPELINE BRONZE
# ==================================================


# Caminho base do container RAW.
# É onde chegam os arquivos Parquet dos micro-lotes
# gerados pelo processo de streaming.
base_raw = (
    f"abfss://raw@{storage_account}.dfs.core.windows.net"
)



# Caminho base do container SQUAD2.
# Dentro dele ficará a organização das camadas
# do projeto (bronze, silver, gold).
base_squad2 = (
    f"abfss://squad2@{storage_account}.dfs.core.windows.net"
)



# ==================================================
# CAMADA RAW
# ==================================================

# Local dos arquivos de origem.
#
# O uso de * permite encontrar todos os micro-lotes:
#
# real-time-data/
#     ano/
#        mês/
#           dia/
#              hora/
#                  ecommerce_pedidos.parquet

caminho_raw_pedidos = (
    f"{base_raw}/real-time-data/*/*/*/*/"
    "ecommerce_pedidos.parquet"
)



# ==================================================
# CAMADA BRONZE
# ==================================================

# Local onde será criada a tabela Delta Bronze.
#
# Estrutura esperada:
#
# grupo3/bronze/ecommerce_pedidos
#
# contendo:
# - _delta_log
# - arquivos parquet
# - partições ano/mês/dia/hora

caminho_bronze_pedidos = (
    f"{base_squad2}/grupo3/bronze/"
    "ecommerce_pedidos"
)



# ==================================================
# CONTROLE DE INGESTÃO
# ==================================================

# Tabela Delta responsável por registrar
# quais arquivos já foram processados.
#
# Evita duplicidade quando o pipeline
# for executado novamente.

caminho_metadata = (
    f"{base_squad2}/grupo3/metadata/"
    "ingestion_control_log"
)



print(
    "Caminhos configurados com sucesso!"
)


print(
    "\nRAW:"
)

print(
    caminho_raw_pedidos
)


print(
    "\nBRONZE:"
)

print(
    caminho_bronze_pedidos
)


print(
    "\nMETADATA:"
)

print(
    caminho_metadata
)

Caminhos configurados com sucesso!

RAW:
abfss://raw@internshipdatalake.dfs.core.windows.net/real-time-data/*/*/*/*/ecommerce_pedidos.parquet

BRONZE:
abfss://squad2@internshipdatalake.dfs.core.windows.net/grupo3/bronze/ecommerce_pedidos

METADATA:
abfss://squad2@internshipdatalake.dfs.core.windows.net/grupo3/metadata/ingestion_control_log


In [0]:
# ==================================================
# FUNÇÃO DE INGESTÃO BRONZE
# ECOMMERCE PEDIDOS
# ==================================================


def ingestao_bronze_pedidos():

    print("====================================")
    print("Iniciando Bronze: ecommerce_pedidos")
    print("====================================")


    # ----------------------------------------------
    # 1. Localizar arquivos Raw disponíveis
    # ----------------------------------------------

    arquivos_raw = [

        item.name

        for item in raw_fs_client.get_paths(
            path="real-time-data",
            recursive=True
        )

        if (
            not item.is_directory
            and item.name.endswith(
                "ecommerce_pedidos.parquet"
            )
        )
    ]


    print(
        "Arquivos encontrados:",
        len(arquivos_raw)
    )


    # ----------------------------------------------
    # 2. Verificar arquivos já processados
    # ----------------------------------------------

    try:

        df_log = (
            spark.read
            .options(**adls_options)
            .format("delta")
            .load(
                caminho_metadata
            )
        )


        arquivos_processados = [

            linha.arquivo_processado

            for linha in df_log.collect()

        ]


    except Exception:

        arquivos_processados = []



    arquivos_novos = [

        arquivo

        for arquivo in arquivos_raw

        if arquivo not in arquivos_processados

    ]


    print(
        "Arquivos novos:",
        len(arquivos_novos)
    )



    if len(arquivos_novos) == 0:

        print(
            "Nenhum arquivo novo encontrado."
        )

        return



    # ----------------------------------------------
    # 3. Ler arquivos Parquet
    # ----------------------------------------------

    dfs = []


    for arquivo in arquivos_novos:


        file_client = (
            raw_fs_client
            .get_file_client(
                arquivo
            )
        )


        conteudo = (
            file_client
            .download_file()
            .readall()
        )


        df_pandas = pd.read_parquet(
            BytesIO(conteudo)
        )


        df_temp = spark.createDataFrame(
            df_pandas
        )


        dfs.append(
            df_temp
        )



    df = dfs[0]


    for temp in dfs[1:]:

        df = df.unionByName(
            temp,
            allowMissingColumns=True
        )



    print(
        "Registros novos:",
        df.count()
    )


    # ----------------------------------------------
    # 4. Adicionar metadados Bronze
    # ----------------------------------------------

    df = (
        df
        .withColumn(
            "bronze_ingested_at",
            F.current_timestamp()
        )
        .withColumn(
            "bronze_source_file",
            F.lit(
                "ecommerce_pedidos.parquet"
            )
        )
    )



    # ----------------------------------------------
    # 5. Criar partições
    # ----------------------------------------------

    df = (
        df
        .withColumn(
            "ano",
            F.year(
                "bronze_ingested_at"
            )
        )
        .withColumn(
            "mes",
            F.month(
                "bronze_ingested_at"
            )
        )
        .withColumn(
            "dia",
            F.dayofmonth(
                "bronze_ingested_at"
            )
        )
        .withColumn(
            "hora",
            F.hour(
                "bronze_ingested_at"
            )
        )
    )



    # ----------------------------------------------
    # 6. Gravar Delta Bronze
    # ----------------------------------------------

    (
        df.write
        .options(**adls_options)
        .format("delta")
        .mode("append")
        .partitionBy(
            "ano",
            "mes",
            "dia",
            "hora"
        )
        .save(
            caminho_bronze_pedidos
        )
    )


    # ----------------------------------------------
    # 7. Registrar arquivos processados
    # ----------------------------------------------

    df_controle = spark.createDataFrame(
        [
            (
                arquivo,
                "ecommerce_pedidos",
                "PROCESSADO"
            )

            for arquivo in arquivos_novos
        ],

        [
            "arquivo_processado",
            "tabela_origem",
            "status"
        ]
    )


    df_controle = (
        df_controle
        .withColumn(
            "data_processamento",
            F.current_timestamp()
        )
    )


    (
        df_controle
        .write
        .options(**adls_options)
        .format("delta")
        .mode("append")
        .save(
            caminho_metadata
        )
    )


    print(
        "Controle de ingestão atualizado!"
    )


    print(
        "Bronze concluída: ecommerce_pedidos"
    )

In [0]:
ingestao_bronze_pedidos()

Iniciando Bronze: ecommerce_pedidos
Arquivos encontrados: 116
Arquivos novos: 116
Registros novos: 774
Controle de ingestão atualizado!
Bronze concluída: ecommerce_pedidos


In [0]:
# ==================================================
# VALIDAR BRONZE FINAL
# ==================================================

bronze_pedidos = (
    spark.read
    .options(**adls_options)
    .format("delta")
    .load(
        caminho_bronze_pedidos
    )
)


print(
    "Quantidade de registros Bronze:",
    bronze_pedidos.count()
)


bronze_pedidos.printSchema()

Quantidade de registros Bronze: 774
root
 |-- id_pedido: long (nullable = true)
 |-- id_cliente: long (nullable = true)
 |-- id_endereco_entrega: long (nullable = true)
 |-- dt_pedido: timestamp (nullable = true)
 |-- status_pedido: string (nullable = true)
 |-- valor_total: double (nullable = true)
 |-- valor_frete: double (nullable = true)
 |-- metodo_pagamento: string (nullable = true)
 |-- dt_previsao_entrega: timestamp (nullable = true)
 |-- dt_ultima_atualizacao_status: timestamp (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- bronze_source_file: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia: integer (nullable = true)
 |-- hora: integer (nullable = true)



In [0]:
ingestao_bronze_pedidos()

Iniciando Bronze: ecommerce_pedidos
Arquivos encontrados: 116
Arquivos novos: 0
Nenhum arquivo novo encontrado.


In [0]:
# ==================================================
# VALIDAR QUANTIDADE FINAL BRONZE
# ==================================================

bronze_pedidos = (
    spark.read
    .options(**adls_options)
    .format("delta")
    .load(
        caminho_bronze_pedidos
    )
)


print(
    "Quantidade de registros Bronze:",
    bronze_pedidos.count()
)


print("\nSchema:")
bronze_pedidos.printSchema()

Quantidade de registros Bronze: 774

Schema:
root
 |-- id_pedido: long (nullable = true)
 |-- id_cliente: long (nullable = true)
 |-- id_endereco_entrega: long (nullable = true)
 |-- dt_pedido: timestamp (nullable = true)
 |-- status_pedido: string (nullable = true)
 |-- valor_total: double (nullable = true)
 |-- valor_frete: double (nullable = true)
 |-- metodo_pagamento: string (nullable = true)
 |-- dt_previsao_entrega: timestamp (nullable = true)
 |-- dt_ultima_atualizacao_status: timestamp (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- bronze_source_file: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia: integer (nullable = true)
 |-- hora: integer (nullable = true)



# ✅ Conclusão

A camada Bronze da tabela `ecommerce_pedidos` foi implementada com sucesso utilizando Delta Lake no Azure Data Lake Gen2.

Durante o processo foram realizadas as seguintes etapas:

- Identificação dos micro-lotes disponíveis na camada Raw;
- Controle de arquivos processados para garantir idempotência da ingestão;
- Leitura dos arquivos `ecommerce_pedidos.parquet`;
- Inclusão dos metadados de auditoria:
  - `bronze_ingested_at`;
  - `bronze_source_file`;
- Criação das partições temporais:
  - ano;
  - mês;
  - dia;
  - hora;
- Gravação dos dados no formato Delta Lake na camada Bronze.

A tabela foi armazenada em:

`grupo3/bronze/ecommerce_pedidos`

A validação final confirmou:

- 774 registros ingeridos;
- estrutura Delta criada com sucesso (`_delta_log`);
- particionamento aplicado corretamente;
- controle de ingestão funcionando, evitando reprocessamento de arquivos já tratados.

A camada Bronze está preparada para alimentar a próxima etapa do pipeline, a camada Silver, onde serão aplicadas regras de qualidade, validações e tratamentos dos dados.